# Mapa de Geoanalitica Territorial - ITT (Ciudad Paraiso / Pulmon de Oriente)

Notebook de geoanalitica territorial. La unidad espacial de analisis es la **manzana catastral** (`data/catastral/MANZANAS_U_2024.shp`). El pipeline:

1. Carga capas catastrales reales (manzanas, terrenos, construcciones) desde `data/catastral/`.
2. Calcula indicadores por manzana e integra los eventos ITT (seguridad / ambiente).
3. Clasifica cada manzana en un nivel ITT (Activacion / Consolidacion / Transformacion / Escala).
4. Exporta artefactos intermedios a `data_processed_geo/` (`.geojson`, `.parquet`).
5. Genera un mapa `.html` interactivo y autonomo (Folium) con popups enriquecidos.

**Convenciones del proyecto:** entorno unico `uv`, codigo sin emojis, paleta Okabe-Ito accesible para daltonismo, CRS metrico EPSG:9377 y capa web EPSG:4326.

> IMPORTANTE - datos de eventos: hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF y censo arboreo NO estan aun en el proyecto. La celda 4 genera datos **sinteticos marcados como placeholder** para que el pipeline sea ejecutable de punta a punta. Al llegar los datos reales, se reemplaza unicamente esa celda.

## 1. Entorno y paleta Okabe-Ito

In [ ]:
# Dependencias: ejecutar en el entorno uv del proyecto.
#   uv run --with geopandas --with folium --with pyarrow jupyter lab
# o, para una celda suelta: uv run --with geopandas --with folium --with pyarrow python ...
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd

# Paleta Okabe-Ito (accesible para daltonismo) - misma paleta que el resto del proyecto
OKI_AZUL     = "#0072B2"
OKI_NARANJA  = "#E69F00"
OKI_VERDE    = "#009E73"
OKI_AMARILLO = "#F0E442"
OKI_AZUL_C   = "#56B4E9"
OKI_BERMELL  = "#D55E00"
OKI_MORADO   = "#CC79A7"
OKI_GRIS     = "#333333"
BG = "#F4F6F9"

# Niveles ITT (color saturado y su version pastel para bandas / coropleta suave)
NIVEL_COLORS = {
    "Activacion":     OKI_BERMELL,
    "Consolidacion":  OKI_NARANJA,
    "Transformacion": OKI_VERDE,
    "Escala":         OKI_AZUL,
}
NIVEL_PASTEL = {
    "Activacion":     "#FFCDD2",
    "Consolidacion":  "#FFE0B2",
    "Transformacion": "#C8E6C9",
    "Escala":         "#BBDEFB",
}
# Color tematico por indicador de evento
EVENTO_COLORS = {
    "hurtos":        OKI_AZUL,
    "siniestros":    OKI_NARANJA,
    "rinas":         OKI_VERDE,
    "spa":           OKI_AMARILLO,
    "lesionados":    OKI_AZUL_C,
    "homicidios":    OKI_BERMELL,
    "vif":           OKI_MORADO,
}

# CRS del proyecto
CRS_WEB    = 4326   # WGS84, capa web
CRS_METRIC = 9377   # MAGNA-SIRGAS Origen Nacional, calculo metrico

print("Entorno listo. Paleta Okabe-Ito cargada.")

## 2. Rutas y carga de capas catastrales (datos reales)

In [ ]:
# Resolucion robusta de rutas: el notebook esta en notebooks/, la data un nivel arriba.
def find_dir(candidatos):
    for p in candidatos:
        if os.path.isdir(p):
            return os.path.abspath(p)
    return os.path.abspath(candidatos[0])

CATASTRAL_DIR = find_dir([os.path.join("..", "data", "catastral"), os.path.join("data", "catastral")])
OUT_DIR = find_dir([os.path.join("..", "data_processed_geo"), "data_processed_geo"])
os.makedirs(OUT_DIR, exist_ok=True)

mz = gpd.read_file(os.path.join(CATASTRAL_DIR, "MANZANAS_U_2024.shp"))
terrenos = gpd.read_file(os.path.join(CATASTRAL_DIR, "TERRENOS_U_2024.shp"))
construcciones = gpd.read_file(os.path.join(CATASTRAL_DIR, "CONSTRUCCION_U_2024.shp"))

# Normalizar clave de union. CONSTRUCCION no trae IDMANZANA: se compone
# como COMUNA(2) + BARRIO(2) + MANZANA(4), igual que en MANZANAS y TERRENOS.
def asegurar_idmanzana(g):
    if "IDMANZANA" not in g.columns:
        g["IDMANZANA"] = (g["COMUNA"].astype(str) + g["BARRIO"].astype(str) + g["MANZANA"].astype(str))
    g["IDMANZANA"] = g["IDMANZANA"].astype(str).str.strip()
    return g

mz = asegurar_idmanzana(mz)
terrenos = asegurar_idmanzana(terrenos)
construcciones = asegurar_idmanzana(construcciones)

print("Catastral:", CATASTRAL_DIR)
print("Salida procesada:", OUT_DIR)
print(f"Manzanas: {len(mz)} | CRS: {mz.crs}")
print(f"Terrenos (predios): {len(terrenos)}")
print(f"Construcciones: {len(construcciones)}")

## 3. Indicadores reales por manzana

In [ ]:
# Area de manzana en metros (reproyectada a CRS metrico)
mz_m = mz.to_crs(CRS_METRIC)
mz = mz.copy()
mz["area_m2"] = mz_m.geometry.area.values

# Predios por manzana
pred_por_mz = terrenos.groupby("IDMANZANA").size().rename("num_predios")

# Construcciones: numero y pisos promedio por manzana
construcciones["NPISOS"] = pd.to_numeric(construcciones.get("NPISOS"), errors="coerce")
const_agg = construcciones.groupby("IDMANZANA").agg(
    num_construcciones=("NPISOS", "size"),
    pisos_prom=("NPISOS", "mean"),
    pisos_max=("NPISOS", "max"),
)

mz = mz.merge(pred_por_mz, on="IDMANZANA", how="left")
mz = mz.merge(const_agg, on="IDMANZANA", how="left")
mz[["num_predios", "num_construcciones"]] = mz[["num_predios", "num_construcciones"]].fillna(0).astype(int)
mz["pisos_prom"] = mz["pisos_prom"].fillna(0).round(2)
mz["pisos_max"] = mz["pisos_max"].fillna(0).astype(int)

# Densidad predial (predios por hectarea)
mz["densidad_predial"] = (mz["num_predios"] / (mz["area_m2"] / 10000.0)).round(1)

mz[["IDMANZANA", "COMUNA", "BARRIO", "area_m2", "num_predios", "num_construcciones", "pisos_prom", "densidad_predial"]].head(10)

## 4. Eventos ITT (PLACEHOLDER - reemplazar con datos reales)

Esta celda es el UNICO punto donde se inyectan datos sinteticos. Cuando lleguen los eventos reales (hurtos, homicidios, rinas, lesionados, siniestralidad, SPA, VIF) y el censo arboreo, reemplazar el contenido por la carga/cruce espacial real a nivel de manzana (join por `IDMANZANA` o join espacial por coordenadas), manteniendo los mismos nombres de columna.

In [ ]:
# ADVERTENCIA: datos sinteticos de demostracion. NO usar para decisiones.
DATOS_SINTETICOS = True
rng = np.random.default_rng(42)
n = len(mz)

# Conteos de eventos por manzana (periodo actual T0 y anterior T-1 para tendencia)
for col in ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]:
    base = rng.integers(0, 12, size=n)
    mz[col + "_t0"] = base
    mz[col + "_t1"] = np.clip(base + rng.integers(-4, 5, size=n), 0, None)

# Censo arboreo (arboles por manzana) y NDVI promedio [0..1]
mz["arboles"] = rng.integers(0, 60, size=n)
mz["ndvi"] = (rng.uniform(0.1, 0.6, size=n)).round(3)

print("Eventos sinteticos generados (placeholder). Columnas de evento:")
print([c for c in mz.columns if c.endswith("_t0") or c in ("arboles", "ndvi")])

## 5. Indice ITT y clasificacion en 4 niveles

In [ ]:
def norm_inv(s):
    """Normaliza 0..1 e invierte: mas eventos negativos -> menor score."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return 1.0 - (s - s.min()) / rng_

def norm_dir(s):
    """Normaliza 0..1 directo: mas es mejor (ej. arbolado)."""
    s = s.astype(float)
    rng_ = s.max() - s.min()
    if rng_ == 0:
        return pd.Series(1.0, index=s.index)
    return (s - s.min()) / rng_

# Dimensiones ITT (ejemplo 5 dim): seguridad, convivencia, movilidad, ambiente, consolidacion urbana
seguridad   = norm_inv(mz["hurtos_t0"] + mz["homicidios_t0"])
convivencia = norm_inv(mz["rinas_t0"] + mz["vif_t0"] + mz["spa_t0"])
movilidad   = norm_inv(mz["siniestros_t0"] + mz["lesionados_t0"])
ambiente    = (norm_dir(mz["arboles"]) + norm_dir(mz["ndvi"])) / 2.0
urbano      = norm_dir(mz["pisos_prom"]) 

mz["itt"] = ((seguridad + convivencia + movilidad + ambiente + urbano) / 5.0 * 100).round(1)

def nivel(itt):
    if itt < 40:  return "Activacion"
    if itt < 60:  return "Consolidacion"
    if itt < 80:  return "Transformacion"
    return "Escala"

mz["nivel"] = mz["itt"].map(nivel)
mz["color_nivel"] = mz["nivel"].map(NIVEL_COLORS)

print(mz["nivel"].value_counts().to_string())
print("\nITT min/med/max:", mz["itt"].min(), mz["itt"].median(), mz["itt"].max())

## 6. Exportar artefactos procesados (data_processed_geo/)

In [ ]:
mz_web = mz.to_crs(CRS_WEB)

geojson_path = os.path.join(OUT_DIR, "manzanas_itt.geojson")
parquet_path = os.path.join(OUT_DIR, "manzanas_itt.parquet")

mz_web.to_file(geojson_path, driver="GeoJSON")
try:
    mz_web.to_parquet(parquet_path)
    print("Parquet:", parquet_path)
except Exception as e:
    print("Parquet no generado (requiere pyarrow):", e)

print("GeoJSON:", geojson_path)
print("Manzanas exportadas:", len(mz_web))

## 7. Popups enriquecidos (HTML/CSS + mini-graficas SVG)

In [ ]:
def svg_barras(valores, colores, w=180, h=70):
    """Mini grafico de barras en SVG inline (sin dependencias JS en el popup)."""
    vmax = max(valores) if max(valores) > 0 else 1
    bw = w / (len(valores) * 1.5)
    gap = bw * 0.5
    barras = []
    for i, (v, c) in enumerate(zip(valores, colores)):
        bh = (v / vmax) * (h - 18)
        x = gap + i * (bw + gap)
        y = h - bh - 12
        barras.append(
            f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" '
            f'fill="{c}" stroke="{OKI_GRIS}" stroke-width="0.5" rx="1"/>'
            f'<text x="{x + bw/2:.1f}" y="{h-2}" font-size="8" fill="{OKI_GRIS}" '
            f'text-anchor="middle">{v}</text>'
        )
    return f'<svg width="{w}" height="{h}" style="background:{BG};border-radius:4px">' + "".join(barras) + "</svg>"

def pct_var(t0, t1):
    if t1 == 0:
        return "s/d" if t0 == 0 else "+100%"
    return f"{(t0 - t1) / t1 * 100:+.0f}%"

def popup_html(r):
    eventos = ["hurtos", "siniestros", "rinas", "spa", "lesionados", "homicidios", "vif"]
    labels = ["Hurtos", "Siniestros", "Rinas", "SPA", "Lesion.", "Homic.", "VIF"]
    vals = [int(r[e + "_t0"]) for e in eventos]
    cols = [EVENTO_COLORS[e] for e in eventos]
    svg = svg_barras(vals, cols)
    filas = "".join(
        f'<tr><td style="padding:1px 6px">{lab}</td>'
        f'<td style="text-align:right;padding:1px 6px"><b>{r[e+"_t0"]}</b></td>'
        f'<td style="text-align:right;padding:1px 6px;color:{OKI_GRIS}">{pct_var(r[e+"_t0"], r[e+"_t1"])}</td></tr>'
        for e, lab in zip(eventos, labels)
    )
    nivel_c = NIVEL_COLORS[r["nivel"]]
    aviso = ('<div style="color:#b00;font-size:9px;margin-top:4px">Datos de evento SINTETICOS (demo)</div>'
             if DATOS_SINTETICOS else "")
    return (
        f'<div style="font-family:system-ui,Segoe UI,sans-serif;font-size:12px;'
        f'background:{BG};padding:8px;border-radius:6px;min-width:230px">'
        f'<div style="font-size:13px;font-weight:700;color:{OKI_GRIS}">Manzana {r["MANZANA"]} '
        f'<span style="font-weight:400;color:#777">({r["IDMANZANA"]})</span></div>'
        f'<div style="color:#555">Comuna {r["COMUNA"]} - Barrio {r["BARRIO"]}</div>'
        f'<div style="margin:4px 0"><span style="background:{nivel_c};color:#fff;'
        f'padding:2px 8px;border-radius:999px;font-size:11px;font-weight:700">'
        f'{r["nivel"]} - ITT {r["itt"]}</span></div>'
        f'<div style="color:#555;font-size:11px">Area: {r["area_m2"]/10000:.2f} ha - '
        f'Predios: {r["num_predios"]} - Pisos prom: {r["pisos_prom"]}</div>'
        f'<div style="margin-top:6px;font-size:10px;color:{OKI_GRIS};font-weight:700">'
        f'Eventos T0 por tipo</div>{svg}'
        f'<table style="font-size:11px;border-collapse:collapse;margin-top:4px;width:100%">'
        f'<tr style="color:{OKI_GRIS};border-bottom:1px solid #ccc">'
        f'<td style="padding:1px 6px">Evento</td><td style="text-align:right;padding:1px 6px">T0</td>'
        f'<td style="text-align:right;padding:1px 6px">vs T-1</td></tr>{filas}</table>{aviso}</div>'
    )

print("Funciones de popup listas. Ejemplo de longitud HTML:", len(popup_html(mz.iloc[0])), "chars")

## 8. Construccion del mapa interactivo (Folium)

In [ ]:
import folium
from folium.plugins import HeatMap

centro = [mz_web.geometry.union_all().centroid.y, mz_web.geometry.union_all().centroid.x]
m = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True)

# 4 capas base conmutables (OSM activa por defecto)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=True).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=False).add_to(m)
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False).add_to(m)

# Capa 1: contorno territorial (activa al inicio)
contorno = folium.FeatureGroup(name="Contorno territorial", show=True)
folium.GeoJson(
    mz_web.to_json(),
    style_function=lambda f: {"color": OKI_GRIS, "weight": 1.2, "fill": False},
).add_to(contorno)
contorno.add_to(m)

# Capa 2: coropleta por nivel ITT (apagada al inicio)
coropleta = folium.FeatureGroup(name="Coropleta ITT (nivel)", show=False)
for _, r in mz_web.iterrows():
    folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=lambda f, c=NIVEL_PASTEL[r["nivel"]], b=NIVEL_COLORS[r["nivel"]]: {
            "fillColor": c, "color": b, "weight": 1.5, "fillOpacity": 0.7},
        popup=folium.Popup(popup_html(r), max_width=280),
        tooltip=f'Manzana {r["MANZANA"]} - {r["nivel"]}',
    ).add_to(coropleta)
coropleta.add_to(m)

print("Mapa base + contorno + coropleta construidos.")

In [ ]:
# Capa 3: simbolos proporcionales de eventos puntuales (ejemplo: hurtos) en el centroide
# NOTA: con datos reales usar las coordenadas reales de cada evento, no el centroide.
sim = folium.FeatureGroup(name="Hurtos (simbolo proporcional)", show=False)
for _, r in mz_web.iterrows():
    cen = r["geometry"].centroid
    val = int(r["hurtos_t0"])
    if val > 0:
        folium.CircleMarker(
            location=[cen.y, cen.x],
            radius=3 + val * 0.9,
            color=OKI_GRIS, weight=0.6,
            fill=True, fill_color=EVENTO_COLORS["hurtos"], fill_opacity=0.75,
            tooltip=f'Hurtos T0: {val} (manzana {r["MANZANA"]})',
        ).add_to(sim)
sim.add_to(m)

# Capa 4: VIF como HeatMap (privacidad: NUNCA punto individual)
vif = folium.FeatureGroup(name="VIF (mapa de calor)", show=False)
pts = []
for _, r in mz_web.iterrows():
    cen = r["geometry"].centroid
    peso = float(r["vif_t0"])
    if peso > 0:
        pts.append([cen.y, cen.x, peso])
if pts:
    HeatMap(pts, radius=28, blur=22, min_opacity=0.3).add_to(vif)
vif.add_to(m)

print("Capas de simbolos proporcionales y HeatMap VIF agregadas. Puntos VIF:", len(pts))

## 9. Leyenda, control de capas y exportacion del HTML

In [ ]:
# Leyenda de niveles ITT (HTML flotante)
items = "".join(
    f'<div style="display:flex;align-items:center;gap:6px;margin:2px 0">'
    f'<i style="width:14px;height:14px;background:{c};border:1px solid {OKI_GRIS};'
    f'display:inline-block;border-radius:3px"></i><span>{k}</span></div>'
    for k, c in NIVEL_COLORS.items()
)
aviso = ('<div style="color:#b00;font-size:10px;margin-top:4px">Eventos: datos sinteticos (demo)</div>'
         if DATOS_SINTETICOS else "")
legend = (
    f'<div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:{BG};'
    f'padding:10px 12px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);'
    f'font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:{OKI_GRIS}">'
    f'<div style="font-weight:700;margin-bottom:4px">Nivel ITT</div>{items}{aviso}</div>'
)
m.get_root().html.add_child(folium.Element(legend))

folium.LayerControl(collapsed=False, position="topright").add_to(m)

html_path = os.path.join(OUT_DIR, "mapa_geoanalitica_itt.html")
m.save(html_path)
print("Mapa exportado:", html_path)
m

## 10. Resumen analitico de hallazgos territoriales

In [ ]:
resumen = {
    "manzanas_analizadas": int(len(mz)),
    "predios_totales": int(mz["num_predios"].sum()),
    "construcciones_totales": int(mz["num_construcciones"].sum()),
    "itt_promedio": float(mz["itt"].mean().round(1)),
    "por_nivel": mz["nivel"].value_counts().to_dict(),
    "datos_eventos_sinteticos": DATOS_SINTETICOS,
}
print(json.dumps(resumen, ensure_ascii=False, indent=2))

print("\nHallazgos (sobre indicadores catastrales reales):")
top_den = mz.sort_values("densidad_predial", ascending=False).head(3)
for _, r in top_den.iterrows():
    print(f"  - Manzana {r['MANZANA']} ({r['IDMANZANA']}): {r['densidad_predial']} predios/ha, "
          f"{r['num_predios']} predios, pisos prom {r['pisos_prom']}")
print("\nNota: la clasificacion ITT depende de los eventos; con datos sinteticos los niveles")
print("son ilustrativos. Los indicadores catastrales (predios, area, pisos, densidad) SI son reales.")